In [ ]:
!pip install -q datasets huggingface_hub pillow matplotlib pandas

In [ ]:
from datasets import load_dataset

dataset = load_dataset("naver-clova-ix/cord-v2")

print(dataset)

Trying to catch the layouts in dataset
 Using embeddings and kmeans clustering
---



In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from transformers import CLIPProcessor, CLIPModel


# ============================================================
# 1. Load CLIP
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

model.eval()


# ============================================================
# 2. Extract image embeddings
# ============================================================

embeddings = []

for i in range(len(dataset["train"])):

    image = dataset["train"][i]["image"].convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        # استخدم vision model مباشرة
        vision_outputs = model.vision_model(
            pixel_values=inputs["pixel_values"]
        )

        # Pooling
        image_features = vision_outputs.pooler_output

        # Project إلى CLIP embedding space
        image_features = model.visual_projection(
            image_features
        )

        # Normalize
        image_features = torch.nn.functional.normalize(
            image_features,
            p=2,
            dim=-1
        )

    embeddings.append(
        image_features.cpu().numpy()[0]
    )

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(dataset['train'])}")


embeddings = np.array(embeddings)

print("\nFinished!")
print("Embedding shape:", embeddings.shape)

In [ ]:
#Clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

results = {}

for k in range(2, 51):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(embeddings)

    score = silhouette_score(
        embeddings,
        labels
    )

    results[k] = score

    print(
        f"Clusters: {k} | "
        f"Silhouette Score: {score:.4f}"
    )

In [ ]:
best_k = max(
    results,
    key=results.get
)

print("Best number of layouts:", best_k)

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(embeddings)

layout_groups = {}

for i, cluster_id in enumerate(clusters):

    layout_groups.setdefault(
        cluster_id,
        []
    ).append(i)


for cluster_id, indexes in layout_groups.items():

    print(
        f"Layout {cluster_id}: "
        f"{len(indexes)} images"
    )

In [ ]:
import matplotlib.pyplot as plt

for cluster_id, indexes in layout_groups.items():

    print("=" * 80)
    print(f"LAYOUT {cluster_id} - {len(indexes)} images")

    sample_indexes = indexes[:8]

    fig, axes = plt.subplots(
        2, 4,
        figsize=(16, 10)
    )

    axes = axes.flatten()

    for ax, idx in zip(axes, sample_indexes):

        image = dataset["train"][idx]["image"]

        ax.imshow(image)
        ax.set_title(f"Index: {idx}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
print(sorted(layout_groups.keys()))

DINOv2 + agglomerative Clustering (vision transformer model )


In [ ]:
!pip install -q transformers scikit-learn

In [ ]:
import torch
import numpy as np

from PIL import Image
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

processor = AutoImageProcessor.from_pretrained(
    "facebook/dinov2-base"
)

model = AutoModel.from_pretrained(
    "facebook/dinov2-base"
).to(device)

model.eval()

print("DINOv2 loaded!")

In [ ]:
embeddings = []

for i in range(len(dataset["train"])):

    image = dataset["train"][i]["image"].convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        #CLS token
        image_embedding = outputs.last_hidden_state[:, 0, :]

        # Normalize
        image_embedding = torch.nn.functional.normalize(
            image_embedding,
            p=2,
            dim=1
        )

    embeddings.append(
        image_embedding.cpu().numpy()[0]
    )

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(dataset['train'])}")


embeddings = np.array(embeddings)

print("\nFinished!")
print("Embedding shape:", embeddings.shape)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.35,
    metric="cosine",
    linkage="average"
)

clusters = clustering.fit_predict(embeddings)

print("Number of detected layouts:", len(set(clusters)))

In [ ]:
layout_groups = {}

for i, cluster_id in enumerate(clusters):

    layout_groups.setdefault(
        cluster_id,
        []
    ).append(i)


print("=" * 60)

for cluster_id, indexes in sorted(layout_groups.items()):

    print(
        f"Layout {cluster_id:02d}: "
        f"{len(indexes)} images"
    )

In [ ]:
import matplotlib.pyplot as plt

for cluster_id, indexes in sorted(layout_groups.items()):

    print("=" * 80)
    print(
        f"LAYOUT {cluster_id} "
        f"({len(indexes)} images)"
    )

    sample_indexes = indexes[:8]

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(16, 10)
    )

    axes = axes.flatten()

    for ax, idx in zip(axes, sample_indexes):

        image = dataset["train"][idx]["image"]

        ax.imshow(image)

        ax.set_title(
            f"Index: {idx}"
        )

        ax.axis("off")


    for ax in axes[len(sample_indexes):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Layout 19
# ============================================================

layout_id = 19

layout_19_indexes = layout_groups[layout_id]

print("Layout 19 indexes:")
print(layout_19_indexes)

print("Number of images:", len(layout_19_indexes))

In [ ]:
layout_19_dataset = dataset["train"].select(layout_19_indexes)

print(layout_19_dataset)

(OCR)

In [ ]:
import matplotlib.pyplot as plt

for i in range(7):

    sample = layout_19_dataset[i]
    image = sample["image"]

    plt.figure(figsize=(10, 14))

    plt.imshow(image)

    plt.title(f"Image {i+1}/7")
    plt.axis("off")

    plt.show()

In [ ]:
for i in range(7):
    sample = layout_19_dataset[i]

    print(f"\n{'='*50}")
    print(f"Ground Truth - Image {i+1}")
    print(f"{'='*50}")

    print(sample["ground_truth"])

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import ImageDraw

for i in range(7):

    sample = layout_19_dataset[i]
    image = sample["image"]

    # Convert ground truth from string to dictionary
    gt = json.loads(sample["ground_truth"])

    img = image.copy()
    draw = ImageDraw.Draw(img)

    for line in gt["valid_line"]:
        for word in line["words"]:

            quad = word["quad"]

            points = [
                (quad["x1"], quad["y1"]),
                (quad["x2"], quad["y2"]),
                (quad["x3"], quad["y3"]),
                (quad["x4"], quad["y4"])
            ]

            draw.line(
                points + [points[0]],
                fill="red",
                width=3
            )

            x = min(p[0] for p in points)
            y = min(p[1] for p in points)

            draw.text(
                (x, max(0, y - 15)),
                line["category"],
                fill="red"
            )

    plt.figure(figsize=(10, 14))
    plt.imshow(img)
    plt.title(f"Ground Truth - Image {i+1}/7")
    plt.axis("off")
    plt.show()

In [ ]:
# At first, see the data before modifying it

import json

for i in range(7):

    sample = layout_19_dataset[i]

    gt = json.loads(sample["ground_truth"])

    print(f"\n{'='*60}")
    print(f"Image {i+1}/7")
    print(f"{'='*60}")

    for line in gt["valid_line"]:

        print(
            "Category:", line["category"],
            "| Text:", " ".join(word["text"] for word in line["words"])
        )

PRODUCT AND SUMMARY SPLITTING (THE REASON FOR SPLIT IS THE NATURE OF RECIEPT $ GROUND TRUTH

In [ ]:
all_products = []

for i in range(7):

    sample = layout_19_dataset[i]
    gt = json.loads(sample["ground_truth"])

    products = {}

    for line in gt["valid_line"]:
        category = line["category"]
        text = " ".join(word["text"] for word in line["words"])
        group_id = line["group_id"]

        if category == "menu.nm":
            products[group_id] = {
                "product": text,
                "quantity": None,
                "price": None
            }

        elif category == "menu.cnt":
            if group_id in products:
                products[group_id]["quantity"] = text

        elif category == "menu.price":
            if group_id in products:
                products[group_id]["price"] = text

    products_list = list(products.values())


    for product in products_list:
        product["image"] = i + 1

    all_products.extend(products_list)


for product in all_products:
    print(product)

In [ ]:
all_summaries = []

for i in range(7):

    sample = layout_19_dataset[i]
    gt = json.loads(sample["ground_truth"])

    summary = {}

    for line in gt["valid_line"]:
        category = line["category"]
        text = " ".join(word["text"] for word in line["words"])

        if category == "sub_total.subtotal_price":
            summary["subtotal"] = text.split()[-1]

        elif category == "sub_total.discount_price":
            summary["discount"] = text.split()[-1]

        elif category == "sub_total.service_price":
            summary["service"] = text.split()[-1]

        elif category == "sub_total.tax_price":
            summary["tax"] = text.split()[-1]

        elif category == "total.total_price":
            summary["total"] = text.split()[-1]


    summary["image"] = i + 1

    all_summaries.append(summary)


for summary in all_summaries:
    print(summary)

POST PROCESSING

In [ ]:
import re


def bbox_info(bbox):
    xs = [int(p[0]) for p in bbox]
    ys = [int(p[1]) for p in bbox]

    return {
        "x1": min(xs),
        "x2": max(xs),
        "y1": min(ys),
        "y2": max(ys),
        "cx": (min(xs) + max(xs)) / 2,
        "cy": (min(ys) + max(ys)) / 2,
        "width": max(xs) - min(xs),
        "height": max(ys) - min(ys)
    }


def is_number(text):
    return bool(re.fullmatch(r"\d+", text.strip()))


def merge_decimal_fragments(result):

    items = []

    # --------------------------------
    # Convert EasyOCR result
    # --------------------------------

    for bbox, text, confidence in result:

        info = bbox_info(bbox)

        items.append({
            "bbox": bbox,
            "text": text.strip(),
            "confidence": float(confidence),
            **info
        })

    # --------------------------------
    # Sort by position
    # --------------------------------

    items.sort(key=lambda x: (x["cy"], x["x1"]))

    used = set()
    output = []

    # --------------------------------
    # NUMBER + suspicious digit + NUMBER
    # --------------------------------

    for i, middle in enumerate(items):

        if i in used:
            continue

        middle_text = middle["text"]

        # Must be exactly one digit
        if not re.fullmatch(r"\d", middle_text):
            continue

        # Low confidence
        if middle["confidence"] >= 0.5:
            continue

        # Tiny width
        if middle["width"] > 30:
            continue

        # --------------------------------
        # Find number on LEFT
        # --------------------------------

        left_candidates = []

        for j, left in enumerate(items):

            if j == i or j in used:
                continue

            if not is_number(left["text"]):
                continue

            # Same line
            y_diff = abs(left["cy"] - middle["cy"])
            max_height = max(
                left["height"],
                middle["height"]
            )

            if y_diff > max_height * 0.35:
                continue

            # Left number must touch / overlap middle
            gap = middle["x1"] - left["x2"]

            if gap <= 15:
                left_candidates.append((j, left))

        # --------------------------------
        # Find number on RIGHT
        # --------------------------------

        right_candidates = []

        for j, right in enumerate(items):

            if j == i or j in used:
                continue

            if not is_number(right["text"]):
                continue

            # Same line
            y_diff = abs(right["cy"] - middle["cy"])
            max_height = max(
                right["height"],
                middle["height"]
            )

            if y_diff > max_height * 0.35:
                continue

            # Right number must touch / overlap middle
            gap = right["x1"] - middle["x2"]

            if gap <= 15:
                right_candidates.append((j, right))

        # --------------------------------
        # Need LEFT + RIGHT
        # --------------------------------

        if not left_candidates or not right_candidates:
            continue

        # Closest left
        left_idx, left = min(
            left_candidates,
            key=lambda x: abs(
                middle["x1"] - x[1]["x2"]
            )
        )

        # Closest right
        right_idx, right = min(
            right_candidates,
            key=lambda x: abs(
                x[1]["x1"] - middle["x2"]
            )
        )

        # --------------------------------
        # Make sure middle is between them
        # --------------------------------

        if left["x2"] > middle["x2"]:
            continue

        if right["x1"] < middle["x1"]:
            continue

        # --------------------------------
        # Middle must be much smaller
        # --------------------------------

        if middle["width"] >= left["width"] * 0.5:
            continue

        if middle["width"] >= right["width"] * 0.5:
            continue

        # --------------------------------
        # CREATE DECIMAL
        # --------------------------------

        new_text = (
            left["text"]
            + "."
            + right["text"]
        )

        new_bbox = [
            [
                left["x1"],
                min(left["y1"], right["y1"])
            ],
            [
                right["x2"],
                min(left["y1"], right["y1"])
            ],
            [
                right["x2"],
                max(left["y2"], right["y2"])
            ],
            [
                left["x1"],
                max(left["y2"], right["y2"])
            ]
        ]

        new_confidence = min(
            left["confidence"],
            right["confidence"]
        )

        # IMPORTANT:
        # Return in EasyOCR format
        output.append((
            new_bbox,
            new_text,
            new_confidence
        ))

        used.add(left_idx)
        used.add(i)
        used.add(right_idx)

    # --------------------------------
    # Add everything NOT merged
    # --------------------------------

    for i, item in enumerate(items):

        if i in used:
            continue

        # IMPORTANT:
        # EasyOCR format
        output.append((
            item["bbox"],
            item["text"],
            item["confidence"]
        ))

    # --------------------------------
    # Sort final result
    # --------------------------------

    output.sort(
        key=lambda detection: (
            min(p[1] for p in detection[0]),
            min(p[0] for p in detection[0])
        )
    )

    return output

In [ ]:
!pip install easyocr

In [ ]:
import easyocr
import numpy as np

reader = easyocr.Reader(['en'])

In [ ]:

all_ocr_results = []

for i in range(7):

    sample = layout_19_dataset[i]
    image = sample["image"]

    image_np = np.array(image.convert("RGB"))

    result = reader.readtext(
        image_np,
        detail=1,
        paragraph=False
    )

    merged_result = merge_decimal_fragments(result)

    all_ocr_results.append(merged_result)

    print(f"Image {i+1}/7 → {len(merged_result)} detections")

In [ ]:
for i, merged_result in enumerate(all_ocr_results, start=1):

    print(f"\n{'='*50}")
    print(f"OCR Result - Image {i}/7")
    print(f"{'='*50}")

    for detection in merged_result:
        print(detection[1])

In [ ]:
for i, merged_result in enumerate(all_ocr_results, start=1):

    print(f"\n{'='*50}")
    print(f"Image {i}/7")
    print(f"{'='*50}")

    for detection in merged_result:
        box = detection[0]
        text = detection[1]
        confidence = detection[2]

        print(f"{text} → Confidence: {confidence:.2%}")

In [ ]:
all_ocr_data = []

for image_number, merged_result in enumerate(all_ocr_results, start=1):

    ocr_data = []

    for detection in merged_result:

        box = detection[0]
        text = detection[1]
        confidence = detection[2]

        center_x = sum(point[0] for point in box) / 4
        center_y = sum(point[1] for point in box) / 4

        ocr_data.append({
            "text": text,
            "x": center_x,
            "y": center_y,
            "confidence": confidence
        })

    ocr_data.sort(key=lambda x: x["y"])

    all_ocr_data.append(ocr_data)

    print(f"Image {image_number}/7 → {len(ocr_data)} detections")

In [ ]:
# Row Detection :::::

all_rows = []

Y_THRESHOLD = 50

for image_number, ocr_data in enumerate(all_ocr_data, start=1):

    rows = []

    for item in ocr_data:

        best_row = None
        best_difference = float("inf")

        for row in rows:

            row_y = sum(x["y"] for x in row) / len(row)
            difference = abs(item["y"] - row_y)

            if difference <= Y_THRESHOLD and difference < best_difference:
                best_row = row
                best_difference = difference

        if best_row is not None:
            best_row.append(item)
        else:
            rows.append([item])

    # Order words horizontally
    for row in rows:
        row.sort(key=lambda x: x["x"])

    # Order rows vertically
    rows.sort(
        key=lambda row: sum(x["y"] for x in row) / len(row)
    )

    # Save rows for this image
    all_rows.append(rows)

    print(f"Image {image_number}/7 → {len(rows)} rows")

CORRECTION THE WORDS AND NUMBERS  & CONVERET THE DATA EXTRACTED INTO DATA FRAME

In [ ]:
# Add x Coordinates

for image_number, rows in enumerate(all_rows, start=1):

    print(f"\n{'='*60}")
    print(f"Image {image_number}/7")
    print(f"{'='*60}")

    for i, row in enumerate(rows):

        print(f"\n--- Row {i+1} ---")

        for item in row:
            print(
                f'Text: {item["text"]:<20} '
                f'X: {item["x"]:.0f} '
                f'Y: {item["y"]:.0f} '
                f'Conf: {item["confidence"]:.2%}'
            )

In [ ]:
import re

ocr_corrections = {
    "Hiriger": "Winger",
    "Hirger": "Winger",
    "Suh": "Sub",
    "Tyta": "Total",
    "Tyta.]": "Total",
    "Tota]": "Total",
    "Tote.": "Total",
    "Kash": "Cash",
    "F'": "",
    "FFrles": "Fries",
    "Pengienaan": "Pengenaan",
    "Pengenaan": "Pengenaan"
}


def correct_common_ocr_errors(text):

    words = text.split()

    corrected_words = []

    for word in words:

        if word in ocr_corrections:
            corrected_words.append(
                ocr_corrections[word]
            )
        else:
            corrected_words.append(word)

    return " ".join(corrected_words)

In [ ]:
def correct_number_text(text):

    text = text.strip()

    # Common OCR substitutions when text is expected to be numeric
    replacements = {
        "O": "0",
        "o": "0",
        "C": "0",
        "c": "0",
        "D": "0",
        "Q": "0"
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove spaces around separators
    text = re.sub(r"\s*,\s*", ",", text)
    text = re.sub(r"\s*\.\s*", ".", text)

    # Remove underscores
    text = text.replace("_", "")

    # Remove spaces inside numbers
    text = re.sub(r"(?<=\d)\s+(?=\d)", "", text)

    return text

In [ ]:
# Word Correction :::::

for image_number, ocr_data in enumerate(all_ocr_data, start=1):

    for item in ocr_data:
        original_text = item["text"]

        corrected_text = correct_common_ocr_errors(original_text)

        item["original_text"] = original_text
        item["text"] = corrected_text

    print(f"Image {image_number}/7 → Word correction applied")

In [ ]:
# Number Correction :::::

for image_number, ocr_data in enumerate(all_ocr_data, start=1):

    for item in ocr_data:

        text = item["text"]

        # نطبق تصحيح الأرقام فقط على النصوص اللي فيها أرقام
        if any(char.isdigit() for char in text):

            corrected_text = correct_number_text(text)

            item["text"] = corrected_text

    print(f"Image {image_number}/7 → Number correction applied")

In [ ]:
for image_number, rows in enumerate(all_rows, start=1):

    print(f"\n{'='*50}")
    print(f"Image {image_number}/7")
    print(f"{'='*50}")

    for i, row in enumerate(rows):

        text = " ".join(item["text"] for item in row)

        print(f"Row {i+1}: {text}")

In [ ]:
import pandas as pd

all_grouped_rows = []

for image_number, rows in enumerate(all_rows, start=1):

    # ==========================================
    # 1. Flatten all OCR detections
    # ==========================================

    detections = []

    for row in rows:
        for item in row:
            detections.append(item)

    # ==========================================
    # 2. Sort by Y position
    # ==========================================

    detections.sort(key=lambda x: x["y"])

    # ==========================================
    # 3. Group detections that are on same line
    # ==========================================

    y_tolerance = 50

    grouped_rows = []

    for item in detections:

        if not grouped_rows:
            grouped_rows.append([item])
            continue

        avg_y = sum(
            x["y"] for x in grouped_rows[-1]
        ) / len(grouped_rows[-1])

        if abs(item["y"] - avg_y) <= y_tolerance:
            grouped_rows[-1].append(item)

        else:
            grouped_rows.append([item])

    # ==========================================
    # 4. Sort each row from left → right
    # ==========================================

    for row in grouped_rows:
        row.sort(key=lambda x: x["x"])

    # ==========================================
    # 5. Save rows for this image
    # ==========================================

    all_grouped_rows.append(grouped_rows)

    # ==========================================
    # 6. Print
    # ==========================================

    print(f"\n{'='*60}")
    print(f"Image {image_number}/7")
    print(f"{'='*60}")

    for i, row in enumerate(grouped_rows, start=1):

        print(f"\n--- Row {i} ---")

        for item in row:
            print(
                f"Text: {item['text']:<20} "
                f"X: {item['x']:.0f} "
                f"Y: {item['y']:.0f} "
                f"Conf: {item['confidence']:.2%}"
            )

In [ ]:
import pandas as pd

data = []

for image_number, rows in enumerate(all_rows, start=1):

    for row_number, row in enumerate(rows, start=1):

        row_text = " ".join(item["text"] for item in row)


        avg_confidence = sum(
            item["confidence"] for item in row
        ) / len(row)


        x_min = min(item["x"] for item in row)
        x_max = max(item["x"] for item in row)

        data.append({
            "image": image_number,
            "row": row_number,
            "text": row_text,
            "x_min": round(x_min, 2),
            "x_max": round(x_max, 2),
            "avg_confidence": round(avg_confidence, 4)
        })

df = pd.DataFrame(data)

df